In [1]:
import tensorflow as tf
import numpy as np
# %load_ext autoreload
# %autoreload 2
import os
import sys
import time
from tensorflow.keras.layers import Dense, Flatten, Dropout
from tensorflow.keras import Model
import pandas as pd
import load_model_compat

I0000 00:00:1793455994.500085   17745 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1793455994.647759   17745 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1793455997.165638   17745 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [2]:
### At this threshold the FPR of the model we trained is 0.1
real_thr = 0.016235964372754097 
### We consider a threshold lower than the real threshold of the model to compensate the impact of group 4 features
### as described in "Towards Evaluation of NIDSs in Adversarial Setting"
thr = 0.016235964372754097*0.8

In [3]:
def get_test_set():
    test_files = ['../data/flow_based/Tuesday-WH-generate-labeled.csv',
                '../data/flow_based/Wednesday-WH-generate-labeled.csv',
                '../data/flow_based/Thursday-WH-generate-labeled.csv',
                '../data/flow_based/Friday-WH-generate-labeled.csv']
    
    train_min = np.load('../data/flow_based/x_train_meta/train_min.npy')
    train_max = np.load('../data/flow_based/x_train_meta/train_max.npy')
    
    x_test_all = []
    y_test_all = []
    all_label_set = []
    for i in range(len(test_files)):
        print (i,test_files[i])
        url_data = test_files[i]
        df = pd.read_csv(url_data)

        feats = df.iloc[:,8:]
        ds_port = df.iloc[:,5]
        df = pd.concat([ds_port,feats],axis=1)

        labels = df.iloc[:,-1].values
        label_set = set(labels)
        all_label_set.append(label_set)

        all_feats = df.iloc[:,:-1].astype(np.float64).values
        known_data_IDs =(np.any(np.isinf(all_feats),axis=1) + np.any(np.isnan(all_feats),axis=1))==False
        x_test = all_feats[known_data_IDs]
        y_test = df.iloc[:,-1].values
        y_test = y_test[known_data_IDs]
#         x_test = (x_test - train_min)/(train_max - train_min+1e-6)
        x_test_all.append(x_test)
        y_test_all.append(y_test)
    x_test = np.concatenate(x_test_all,axis=0).astype(np.float32)
    y_test = np.concatenate(y_test_all,axis=0)
    
    #### list of features which are decimal:
    decimal_features = []
    for i in range(x_test.shape[1]):
        a1 = x_test[:,i]
        a2 = np.round(a1)
        temp = np.sum(np.abs(a1-a2))
        if temp==0:
#             print (i,df.columns[i])
            decimal_features.append(i)
    return x_test, y_test,train_min, train_max,decimal_features
    
    

In [4]:
@tf.function
def test_step(x):
    def_mask = tf.random.uniform(shape=[1*100,num_input])
    def_mask = tf.cast((def_mask>0.75),tf.float32)
    x_normalized =(x - train_min)/(train_max - train_min+0.000001)

    partial_x = def_mask*x_normalized
    rec_x = model(partial_x, training=False)
    score = tf.reduce_mean(tf.square(rec_x - x_normalized),axis=1)
    score = tf.reduce_min(tf.reshape(score,[5,20]),axis=-1)
    score = tf.reduce_sum(score)
    return score

In [5]:
model = load_model_compat.load_model('../models/flw_model/')

x_test, y_test, train_min, train_max, decimal_features = get_test_set()
num_input = x_test.shape[1]


I0000 00:00:1793456001.938171   17745 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 1767 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 2050, pci bus id: 0000:01:00.0, compute capability: 8.6


0 ../data/flow_based/Tuesday-WH-generate-labeled.csv
1 ../data/flow_based/Wednesday-WH-generate-labeled.csv
2 ../data/flow_based/Thursday-WH-generate-labeled.csv
3 ../data/flow_based/Friday-WH-generate-labeled.csv


In [6]:
label_names = ['BENIGN','FTP-Patator','SSH-Patator','DoS slowloris','DoS Slowhttptest','DoS Hulk','DoS GoldenEye',
          'Heartbleed','Web Attack','Infiltration', 'Bot', 'PortScan', 'DDoS']

In [7]:
attack_type = 'DDoS'

In [8]:
x_test_mal = x_test[y_test==attack_type]
print (x_test_mal.shape)
x_test_mal = x_test_mal[:5000].astype(np.float32)
score_np = np.zeros(len(x_test_mal))
begin_time = time.time()
for i in range(len(x_test_mal)):
    if i%1000==0:
        print (i,time.time() - begin_time)
    sample = x_test_mal[i][None]
    score_temp = test_step(sample)
    score_np[i] = score_temp.numpy()
print (i,time.time() - begin_time)
normal_tpr = np.sum(score_np>=real_thr)/len(score_np)
print ("TPR in normal setting for "+attack_type+" is {0:0.4f}".format(normal_tpr))

(128025, 77)
0 0.00010323524475097656
1000 8.542198419570923
2000 13.946751356124878
3000 15.801787376403809
4000 19.40602469444275
4999 23.421534776687622
TPR in normal setting for DDoS is 0.5772


# Crafting Adversarial Examples

In [9]:
alpha = tf.Variable(np.zeros((1, x_test.shape[1]), dtype=np.float32),name='modifier')

In [10]:
@tf.function
def get_adv(x,adv_mask):
    x_normalized = (x - train_min)/(train_max - train_min+1e-6)
    alpha_masked = alpha*adv_mask
    adv_ex_normalized = x_normalized + alpha_masked
    adv_ex = adv_ex_normalized*(train_max - train_min + 1e-6) + train_min #### unnormalized
    return adv_ex

In [11]:
@tf.function
def optim(x,adv_mask,optimizer):
    with tf.GradientTape() as tape:
        x_normalized = (x - train_min)/(train_max - train_min+1e-6)
        alpha_masked = alpha*adv_mask
        adv_ex_normalized = x_normalized + alpha_masked
    
        def_mask = tf.random.uniform(shape=[1*100,num_input])
        def_mask = tf.cast((def_mask>0.75),tf.float32)
        partial_x = def_mask*adv_ex_normalized
        rec_x = model(partial_x, training=False)

        score = tf.reduce_mean(tf.square(rec_x - adv_ex_normalized),axis=1)
        score = tf.reduce_sum(score)
        loss = score

    gradients = tape.gradient(loss, [alpha])
    optimizer.apply_gradients(zip(gradients, [alpha]))

In [12]:
fixed_mask = np.ones(77,dtype=np.float32)

######### Group 1 features (based on the categorization in  "Towards Evaluation of NIDSs in Adversarial Setting") ######### 
fixed_mask[0] = 0 #dst_port
fixed_mask[3] = 0 #bwd
fixed_mask[4] = 0 #total len of fwd pkts (sum of all payloads in fwd direction)
fixed_mask[5] = 0 #total len of bwd pkts (sum of all payloads in bwd direction)
fixed_mask[10] = 0 #bwd
fixed_mask[11] = 0 #bwd
fixed_mask[12] = 0 #bwd
fixed_mask[13] = 0 #bwd
fixed_mask[25] = 0 #bwd
fixed_mask[26] = 0 #bwd
fixed_mask[27] = 0 #bwd
fixed_mask[28] = 0 #bwd
fixed_mask[29] = 0 #bwd
fixed_mask[31] = 0 #bwd
fixed_mask[33] = 0 #bwd
fixed_mask[35] = 0 #bwd
fixed_mask[37] = 0 #bwd
fixed_mask[53] = 0 
fixed_mask[54] = 0 #bwd
for i in range(58,65):
    fixed_mask[i]=0
fixed_mask[66] = 0 #bwd



######### Group 4 features (based on the categorization in  "Towards Evaluation of NIDSs in Adversarial Setting") ######### 
fixed_mask[9] = 0 #std (fwd-payload)
fixed_mask[17] = 0 #std
fixed_mask[22] = 0 #std
fixed_mask[41] = 0 #std
fixed_mask[42] = 0 #var
fixed_mask[55] = 0 #bulk
fixed_mask[56] = 0 #bulk
fixed_mask[57] = 0 #bulk
for i in range(68,77):
    fixed_mask[i]=0
    

    
################## Group 3 (Dependent) features (based on the categorization in  "Towards Evaluation of NIDSs in Adversarial Setting") ######### 
depended_features = {8,14,15,16,20,21,34,36,38,39,40,51,52,67}
for i in depended_features:
    fixed_mask[i] = 0
    
    
################## Other features are Group 2 (Independent) features

In [13]:
mask_l1 = np.copy(fixed_mask)
mask_l2 = np.copy(fixed_mask)
####unmask stds
mask_l2[9]=1
mask_l2[17]=1
mask_l2[22]=1
mask_l2[41]=1
mask_l2[42]=1
mask_l3 = np.copy(mask_l2)
mask_l3[55] = 1 #bulk
mask_l3[56] = 1 #bulk
mask_l3[57] = 1 #bulk
for i in range(68,77):
    mask_l3[i]=1

In [14]:
#### The duplicated features:
dup_ht={61:2,
        63:3,
        62:4,
        64:5}
mins = [7,19,24,38]
maxs = [6,18,23,39]
flags = [30,32,43,44,45,46,47,48,49,50]
aggr_features=[9,17,22,41,42,55,56,57,68,69,70,71,72,73,74,75,76]

In [15]:
optimizer_001 = tf.keras.optimizers.Adam(learning_rate=0.001)
optimizer_01 = tf.keras.optimizers.Adam(learning_rate=0.01)
optimizer_1 = tf.keras.optimizers.Adam(learning_rate=0.1)
optimizer_10 = tf.keras.optimizers.Adam(learning_rate=1.)
all_optimizers = [optimizer_001,optimizer_01,optimizer_1,optimizer_10]
for op in all_optimizers:
    op.apply_gradients(zip([alpha],[alpha]))

In [16]:
def find_adv(source_index):
    alpha.assign(np.zeros(alpha.shape))
    ##### CHECK TO SEE IF IT NEEDS TO BE ADVERSARIALLY CHANGED #####
    orig_sample = np.copy(x_test_mal[source_index][None])
    adv_ex = get_adv(orig_sample,mask_l1)
    sc = test_step(adv_ex)
    sc = sc.numpy()
    if sc<thr:
        return 'no change'
    mal_counter[0]+=1
    backup_adv = [None]
    def optimize(optimizer,total_iter,mask_v):
        alpha.assign(np.zeros(alpha.shape))
        for i in range(total_iter):
            optim(orig_sample,mask_v,optimizer)
            adv_ex = get_adv(orig_sample,mask_v)
            adv_ex_np = adv_ex.numpy()
            for k in dup_ht:
                adv_ex_np[0,k] = adv_ex_np[0,dup_ht[k]]
            adv_ex_np[0,mins] = np.maximum(0,adv_ex_np[0,mins])
            adv_ex_np[0,mins] = np.minimum(orig_sample[0,mins],adv_ex_np[0,mins])
            adv_ex_np[0,maxs] = np.maximum(orig_sample[0,maxs],adv_ex_np[0,maxs])
            adv_ex_np[0,flags] = np.maximum(orig_sample[0,flags],adv_ex_np[0,flags])
            flags_max_changed = np.max(adv_ex_np[0,flags] - orig_sample[0,flags])
            adv_ex_np[0,aggr_features] = np.maximum(0,adv_ex_np[0,aggr_features])
            adv_ex_np[0,2] = np.maximum(orig_sample[0,2]+flags_max_changed,adv_ex_np[0,2])
            adv_ex_np[0,65] = np.maximum(0,adv_ex_np[0,65])
            
            ##### round the ones that should be rounded #### 
            adv_ex_np[0,decimal_features] = np.round(adv_ex_np[0,decimal_features])
            ##################### recalculate dependent features ######################
            adv_ex_np[0,4] = adv_ex_np[0,4] + (adv_ex_np[0,6]!=orig_sample[0,6])*adv_ex_np[0,6] + (adv_ex_np[0,7]!=orig_sample[0,7])*adv_ex_np[0,7]
            adv_ex_np[0,8] = adv_ex_np[0,4]/adv_ex_np[0,2]
            adv_ex_np[0,14]=(adv_ex_np[0,4]+adv_ex_np[0,5])/adv_ex_np[0,1]*1e6
            adv_ex_np[0,15]=(adv_ex_np[0,2]+adv_ex_np[0,3])/adv_ex_np[0,1]*1e6
            adv_ex_np[0,16]=adv_ex_np[0,1]/(adv_ex_np[0,2]+adv_ex_np[0,3]-1)
            
            adv_ex_np[0,21]=adv_ex_np[0,20]/(adv_ex_np[0,2]-1)
            adv_ex_np[0,34]=adv_ex_np[0,34] + 20*(adv_ex_np[0,2] - orig_sample[0,2])
            adv_ex_np[0,36]=adv_ex_np[0,2]/adv_ex_np[0,1]*1e6
            
            adv_ex_np[0,38]=np.minimum(adv_ex_np[0,38],adv_ex_np[0,7])
            adv_ex_np[0,39]=np.maximum(adv_ex_np[0,39],adv_ex_np[0,6])
            
            adv_ex_np[0,40]=(adv_ex_np[0,4]+adv_ex_np[0,5])/(adv_ex_np[0,3]+adv_ex_np[0,2]+1)
            adv_ex_np[0,51]=adv_ex_np[0,3]/adv_ex_np[0,2]
            adv_ex_np[0,52]=(adv_ex_np[0,4]+adv_ex_np[0,5])/(adv_ex_np[0,3]+adv_ex_np[0,2])
            adv_ex_np[0,53]=adv_ex_np[0,8]
            adv_ex_np[0,67]=adv_ex_np[0,67] + (adv_ex_np[0,6]!=orig_sample[0,6])*1 + (adv_ex_np[0,7]!=orig_sample[0,7])*1
            
            adv_ex_np[np.isinf(adv_ex_np)]=0
            adv_ex_np[np.isnan(adv_ex_np)]=0
            sc = test_step(adv_ex_np)
            sc = sc.numpy()
            if sc<thr:
                return adv_ex_np
            if sc<real_thr and isinstance(backup_adv[0],type(None)):
                backup_adv[0] = adv_ex_np
                
        return None

    for mask in [mask_l1,mask_l2,mask_l3]:
        res = optimize(optimizer_001,30,mask)
        if isinstance(res,type(None)):
            res = optimize(optimizer_01,40,mask)
            if isinstance(res,type(None)):
                res = optimize(optimizer_1,50,mask)
                if isinstance(res,type(None)):
                    res = optimize(optimizer_10,60,mask)
        if isinstance(res,type(None))==False:
            break
    if isinstance(res,type(None)):
        return backup_adv[0]
    return res


In [37]:
# from gpu_adversarial import GpuAdversarialSearch

# gpu_search = GpuAdversarialSearch(
#     model=model,
#     train_min=train_min,
#     train_max=train_max,
#     decimal_features=decimal_features,
#     duplicate_features=dup_ht,
#     mins=mins,
#     maxs=maxs,
#     flags=flags,
#     aggregate_features=aggr_features,
#     threshold=thr,
#     real_threshold=real_thr,
# )
# gpu_optimizers = [
#     tf.keras.optimizers.Adam(learning_rate=0.001),
#     tf.keras.optimizers.Adam(learning_rate=0.01),
#     tf.keras.optimizers.Adam(learning_rate=0.1),
#     tf.keras.optimizers.Adam(learning_rate=1.0),
# ]

# def find_adv(source_index):
#     original = np.asarray(x_test_mal[source_index][None], dtype=np.float32)
#     return gpu_search.find(
#         original=original,
#         masks=[mask_l1, mask_l2, mask_l3],
#         optimizers=gpu_optimizers,
#         iterations=[30, 40, 50, 60],
#     )


In [17]:
x_test_mal = x_test[y_test==attack_type]
x_test_mal = x_test_mal[:5000].astype(np.float32)
x_test_adv = np.zeros_like(x_test_mal)
mal_counter = [0]
cons_as_mal = 0
cons_as_ben = 0
fooled = 0
st = time.time()
print (attack_type, x_test_mal.shape)
for i in range(len(x_test_mal)):
    if i%50==0:
        print ('%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%',i,time.time()-st,cons_as_mal,cons_as_ben,fooled)
    res = find_adv(i)
    if isinstance(res,str) and res=='no change':
        cons_as_ben+=1
        x_test_adv[i] = np.copy(x_test_mal[i])
    elif isinstance(res,type(None)):
        cons_as_mal+=1
        x_test_adv[i] = np.copy(x_test_mal[i])
    else:
        fooled+=1
        x_test_adv[i] = res
execution_time = time.time()-st
print ('%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%',i,time.time()-st,cons_as_mal,cons_as_ben,fooled,execution_time)

DDoS (5000, 77)
%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%% 0 0.0003402233123779297 0 0 0
%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%% 50 132.1159381866455 25 24 1
%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%% 100 260.446325302124 50 49 1
%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%% 150 390.20633578300476 75 74 1
%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%% 200 521.9670295715332 100 99 1
%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%% 250 652.2613191604614 126 123 1
%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%% 300 807.5965552330017 156 143 1
%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%% 350 950.9362545013428 185 150 15
%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%% 400 1085.0081782341003 211 174 15
%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%% 450 1229.5247311592102 237 196 17
%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%% 500 1398.4992735385895 270 212 18
%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%% 550 1529.2909054756165 296 236 18
%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%% 600 1687.0129554271698 326 255 19
%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%% 650 1831.5861458778381 351 279 20
%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%% 700 1993.808814048767 383 296 21
%%%%%%%%%%%%%%%%%%%%

/tmp/ipykernel_17745/1138139206.py:38: RuntimeWarning: invalid value encountered in scalar divide
  adv_ex_np[0,21]=adv_ex_np[0,20]/(adv_ex_np[0,2]-1)


%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%% 3600 10915.1949737072 2042 1437 121
%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%% 3650 11064.476420164108 2069 1458 123
%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%% 3700 11228.833615779877 2099 1478 123
%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%% 3750 11374.246655225754 2126 1500 124
%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%% 3800 11524.939383983612 2156 1520 124
%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%% 3850 11708.35932135582 2192 1534 124
%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%% 3900 11842.832567691803 2218 1558 124
%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%% 3950 11993.835983753204 2246 1579 125
%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%% 4000 12125.330034255981 2270 1603 127
%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%% 4050 12273.124960899353 2299 1619 132
%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%% 4100 12460.182064533234 2335 1625 140
%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%% 4150 12590.507289171219 2360 1649 141
%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%% 4200 12732.456609249115 2388 1671 141
%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%% 4250 12867.096918821335 2414 1694 142
%%%%%%%%%%%%%%%%%%%%%%%

In [18]:
print ("TPR of the attacker's local copy of the NIDS: {0:0.4f}".format(cons_as_mal/len(x_test_mal)))

TPR of the attacker's local copy of the NIDS: 0.5662


In [19]:
score_np2 = np.zeros(len(x_test_adv))
begin_time = time.time()
for i in range(len(x_test_adv)):
    if i%1000==0:
        print (i,time.time() - begin_time)
    sample = x_test_adv[i][None]
    score_temp = test_step(sample)
    score_np2[i] = score_temp.numpy()
print (i,time.time() - begin_time)
mal_scores = score_np2
print ("TPR of the victim's NIDS: {0:0.4f}".format(np.sum(mal_scores>=thr)/(0. + len(mal_scores))))

0 0.00018453598022460938
1000 3.0089752674102783
2000 5.7066826820373535
3000 8.721267461776733
4000 11.534685850143433
4999 14.28234052658081
TPR of the victim's NIDS: 0.5884


In [20]:
from experiment_results import ExperimentResult, save_result_json
results_path = "../results/flow_based_adversarial_results.json"
save_result_json(
    ExperimentResult(
        best_threshold=float(thr),
        victim_tpr=float(np.sum(mal_scores >= thr) / len(mal_scores)),
        attacker_tpr=float(cons_as_mal / len(x_test_mal)),
        total_runtime_seconds=float(execution_time),
        attack_type=str(attack_type),
        model_type="flow",
        normal_tpr=normal_tpr,
    ),
    results_path,
)
print(f"Adversarial result saved to {results_path}")

Adversarial result saved to ../results/flow_based_adversarial_results.json
